# Remove dark sprite sheet backgrounds

This notebook removes near-black backgrounds from a sprite sheet and saves a transparent PNG.

Use it when a sprite sheet has a solid black or very dark backdrop that should become transparent for in-game rendering.

In [ ]:
from pathlib import Path

from PIL import Image
from IPython.display import display


def remove_dark_background(input_path, output_path=None, darkness_threshold=40, alpha_falloff=20):
    """Convert near-black pixels to transparency and save the cleaned sprite sheet."""
    input_path = Path(input_path)
    if output_path is None:
        output_path = input_path.with_name(f"{input_path.stem}_transparent.png")
    else:
        output_path = Path(output_path)

    image = Image.open(input_path).convert("RGBA")
    pixels = image.load()
    width, height = image.size

    for y in range(height):
        for x in range(width):
            red, green, blue, alpha = pixels[x, y]
            darkest_channel = max(red, green, blue)

            if darkest_channel <= darkness_threshold:
                pixels[x, y] = (red, green, blue, 0)
            elif darkest_channel <= darkness_threshold + alpha_falloff:
                fade_ratio = (darkest_channel - darkness_threshold) / alpha_falloff
                pixels[x, y] = (red, green, blue, int(alpha * fade_ratio))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path)
    return output_path


def preview_image(image_path):
    display(Image.open(image_path))